# 📊 Fase 2: Exploratory Data Analysis (EDA)

## Objetivo

Una vez integrados los datasets principales, el siguiente paso consiste en comprender el comportamiento del negocio a través de los datos.

El Análisis Exploratorio de Datos (EDA) nos permitirá:

- Evaluar la calidad de los datos.
- Identificar valores faltantes y posibles anomalías.
- Comprender la distribución de las ventas.
- Analizar patrones temporales.
- Identificar los productos y tiendas más importantes.
- Estudiar el impacto de promociones, festivos y otras variables externas.

## Preguntas de Negocio

Durante esta fase se va responder lo siguiente:

1. ¿Cómo evolucionan las ventas a lo largo del tiempo?
2. ¿Qué tiendas generan mayores ventas?
3. ¿Qué familias de productos son más importantes?
4. ¿Las promociones aumentan las ventas?
5. ¿Existen patrones por día de la semana o por mes?
6. ¿Los festivos afectan el comportamiento de compra?
7. ¿Existe relación entre las ventas y factores externos?

# 🔍 2.1 Calidad de los Datos

Antes de realizar cualquier análisis es necesario verificar la calidad de la información.

En esta sección revisaremos:

- Tipos de datos.
- Valores faltantes.
- Registros duplicados.
- Consistencia general del dataset.

In [ ]:
# carga de librerias

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# configuracion de pandas de mostrar todas las columnas
pd.set_option("display.max_columns", None)

# configuracion de pandas decimales a mostrar
pd.set_option("display.float_format", "{:,.2f}".format)

# direccion de los datos
PATH = "../data/raw/"

In [2]:
# Carga de datos

train = pd.read_csv(PATH + "train.csv")

stores = pd.read_csv(PATH + "stores.csv")

items = pd.read_csv(PATH + "items.csv")

/var/folders/c5/ylt_kbzs0k5frnlwl06ztm480000gn/T/ipykernel_2650/258580417.py:3: DtypeWarning: Columns (0: onpromotion) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(PATH + "train.csv")


# Dataset `train`

## Creación del conjunto de trabajo

Para preservar la integridad del conjunto de datos original, se crea una copia denominada `train_clean`. A partir de este punto, todas las tareas de limpieza, transformación y análisis exploratorio se realizarán sobre esta copia.

**Objetivo:** mantener el dataset `train` sin modificaciones para poder utilizarlo como referencia durante todo el proyecto.

In [3]:
# copia de los datos limpios
train_clean = train.copy()

In [4]:
# Exploracion de los datos
train_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 125497040 entries, 0 to 125497039
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   id           int64  
 1   date         str    
 2   store_nbr    int64  
 3   item_nbr     int64  
 4   unit_sales   float64
 5   onpromotion  object 
dtypes: float64(1), int64(3), object(1), str(1)
memory usage: 5.6+ GB


## Diccionario de Variables

| Columna | Descripción | Tipo de dato |
|----------|------------|-------------|
| id | Identificador único del registro de venta | `int64` |
| date | Fecha de la venta | `str` |
| store_nbr | Identificador de la tienda | `int64` |
| item_nbr | Identificador del producto | `int64` |
| unit_sales | Cantidad de unidades vendidas | `float64` |
| onpromotion | Indica si el producto estaba en promoción (True/False) | `object` |

### Observaciones

- **date** se encuentra como texto (`str`) y deberá convertirse a formato fecha (`datetime`).
- **onpromotion** presenta valores nulos y posteriormente se evaluará su conversión a tipo booleano.
- **store_nbr** e **item_nbr** son identificadores categóricos representados numéricamente.
- **unit_sales** es la variable objetivo del análisis y representa la cantidad de unidades vendidas.


In [5]:
train_clean.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


## Validación de rangos en variables numéricas

Como parte del proceso de calidad de datos, se revisan los valores mínimos y máximos de las variables numéricas.

El objetivo es verificar que los datos se encuentren dentro de rangos razonables para el negocio y detectar posibles anomalías, tales como:

- Ventas negativas inesperadas.
- Identificadores inválidos.
- Valores extremadamente altos o bajos.
- Registros potencialmente erróneos.

Esta revisión permite identificar tempranamente problemas que podrían afectar el análisis exploratorio y el desempeño de los modelos predictivos.

In [ ]:
# Exploracion de los datos
train_clean.describe()

,id,store_nbr,item_nbr,unit_sales
count,1.254970e+08,1.254970e+08,1.254970e+08,1.254970e+08
mean,6.274852e+07,2.746458e+01,9.727692e+05,8.554865e+00
std,3.622788e+07,1.633051e+01,5.205336e+05,2.360515e+01
min,0.000000e+00,1.000000e+00,9.699500e+04,-1.537200e+04
25%,3.137426e+07,1.200000e+01,5.223830e+05,2.000000e+00
50%,6.274852e+07,2.800000e+01,9.595000e+05,4.000000e+00
75%,9.412278e+07,4.300000e+01,1.354380e+06,9.000000e+00
max,1.254970e+08,5.400000e+01,2.127114e+06,8.944000e+04


## Análisis de las variables numéricas

A continuación se presentan algunas observaciones obtenidas a partir de las estadísticas descriptivas de las variables numéricas del conjunto de datos.

### Observaciones

- **id**
  - Es un identificador único para cada registro.
  - Sus valores van desde **0** hasta **125.496.969**, lo que coincide con el número total de observaciones.
  - No aporta información predictiva para el modelo, por lo que posteriormente podrá eliminarse.

- **store_nbr**
  - Existen **54 tiendas**, identificadas con valores entre **1** y **54**.
  - Aunque está almacenada como un número entero (`int64`), representa una **variable categórica** y no una magnitud numérica.

- **item_nbr**
  - Los identificadores de los productos varían entre **96.995** y **2.127.114**.
  - Al igual que `store_nbr`, corresponde a una **variable categórica codificada numéricamente**, por lo que su valor no representa una cantidad.

- **unit_sales**
  - La cantidad promedio vendida es de **8.55 unidades** por registro.
  - La mediana es de **4 unidades**, mientras que el tercer cuartil es de **9 unidades**, indicando que la mayoría de las ventas corresponden a pocas unidades por transacción.
  - Se observan **valores negativos**, con un mínimo de **-15.372**, lo que podría corresponder a devoluciones, cancelaciones o ajustes de inventario.
  - También existen valores extremadamente altos, alcanzando un máximo de **89.440 unidades**, los cuales deberán revisarse para determinar si corresponden a ventas reales o posibles valores atípicos.

### Conclusiones

- Las variables `store_nbr` e `item_nbr` deben tratarse como variables categóricas durante el análisis y el modelado.
- La variable `id` funciona únicamente como identificador y no será útil como variable predictora.
- La variable `unit_sales` requiere un análisis más detallado para estudiar la presencia de devoluciones y posibles valores atípicos antes de entrenar modelos de predicción.

In [7]:
#ver valores negativos de unit_sales
train_clean[train_clean["unit_sales"] < 0]

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
10655,10655,2013-01-02,10,456875,-3.0,NaN
46867,46867,2013-01-03,5,559044,-1.0,NaN
50970,50970,2013-01-03,9,365138,-3.0,NaN
71807,71807,2013-01-03,41,812716,-19.0,NaN
71992,71992,2013-01-03,41,1004551,-27.0,NaN
...,...,...,...,...,...,...
125401397,125401397,2017-08-15,4,682884,-1.0,False
125404116,125404116,2017-08-15,5,1455485,-1.0,False
125407220,125407220,2017-08-15,7,215303,-1.0,False
125409295,125409295,2017-08-15,7,2048193,-1.0,False


In [8]:
# Valores nulos
train_clean.isnull().sum()

id                    0
date                  0
store_nbr             0
item_nbr              0
unit_sales            0
onpromotion    21657651
dtype: int64

## Análisis de valores nulos

En esta etapa se identifica la cantidad de valores faltantes en cada variable del conjunto de datos. La presencia de datos nulos puede afectar el análisis exploratorio, el preprocesamiento y el entrenamiento de modelos de Machine Learning.

### Objetivos de esta revisión

- Identificar variables con información faltante.
- Cuantificar la magnitud de los valores nulos.
- Determinar si los datos faltantes requieren tratamiento.
- Evaluar el posible impacto sobre el análisis y la construcción del modelo predictivo.

### Observaciones

- Las variables `id`, `date`, `store_nbr`, `item_nbr` y `unit_sales` no presentan valores nulos.
- La variable `onpromotion` contiene **21.657.651** valores faltantes, por lo que será necesario analizar el significado de estos registros antes de decidir cómo tratarlos.
- No es recomendable eliminar los registros únicamente por presentar valores nulos en `onpromotion`, ya que representan una parte importante del conjunto de datos.

### Conclusión

El conjunto de datos presenta una buena calidad en la mayoría de sus variables. La única variable con datos faltantes es `onpromotion`, la cual será analizada posteriormente para determinar la estrategia de tratamiento más adecuada.

In [9]:
# ver los valores nulos
train_clean[train_clean["onpromotion"].isnull()]

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.000,NaN
1,1,2013-01-01,25,105574,1.000,NaN
2,2,2013-01-01,25,105575,2.000,NaN
3,3,2013-01-01,25,108079,1.000,NaN
4,4,2013-01-01,25,108701,1.000,NaN
...,...,...,...,...,...,...
21657646,21657646,2014-03-31,54,1696039,61.386,NaN
21657647,21657647,2014-03-31,54,1696042,6.248,NaN
21657648,21657648,2014-03-31,54,1696047,16.810,NaN
21657649,21657649,2014-03-31,54,1696051,23.000,NaN


## Verificación de registros duplicados

Se realizó una verificación para identificar registros duplicados en el conjunto de datos utilizando el método `duplicated()`.

La revisión confirmó que **no existen registros duplicados**, por lo que no fue necesario eliminar observaciones.

> **Nota:** El código utilizado para esta validación no se incluye en el notebook, ya que su ejecución sobre un conjunto de datos de este tamaño consume una cantidad considerable de tiempo y memoria. La comprobación se realizó una única vez al inicio del proyecto para garantizar la calidad de los datos.

# Correcion tipos de datos

# Transofrmar la columna fecha de str a `date.time` 

In [ ]:
# convertir la columna date a tipo fecha

train_clean["date"] = pd.to_datetime(train_train["date"])

posteriormente se crearan otras columnas para el año mes y dia para poder realizar mejores analisis.

# se van a tratar los nulos en onpromotion y se pasaran a False pues a no tener informacion se asume que en su momento el producto no estaba en promocion

In [ ]:
# convertir los nulos de onpromotion a False
train_clean["onpromotion"] = train_clean["onpromotion"].fillna(False)

# cambiar el tipo de dato de onpromotion a booleano
train_clean["onpromotion"] = train_clean["onpromotion"].astype(bool)

una vez finalizado la exploracion con los datos de `train` pocedemos a los de `stores`

# Dataset `stores`

## Creación del conjunto de trabajo

Para preservar la integridad del conjunto de datos original, se crea una copia denominada `stores_clean`. A partir de este punto, todas las tareas de limpieza, transformación y análisis exploratorio se realizarán sobre esta copia.

**Objetivo:** mantener el dataset `stores` sin modificaciones para poder utilizarlo como referencia durante todo el proyecto.